# Lesson 2 · Embeddings: turning text into numbers

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- see why we represent texts as vectors (lists of numbers)
- measure how similar two vectors are with **cosine similarity**, by hand
- open up a real embedding model and follow a sentence through it, step by step
- embed real MAGE texts and look at them in a picture

**Runtime:** a CPU works; a GPU is faster (Runtime → Change runtime type → T4 GPU).

## 1. The idea, with a toy example

Computers can't compare sentences directly, but they are very good at comparing numbers. So we describe each text with a list of numbers, called a **vector** or **embedding**, chosen so that *similar texts get similar vectors*.

To see the idea, let's invent three "features" and score each sentence by hand:

| | formal? | emotional? | about science? |
|---|---|---|---|
| "The results indicate a significant improvement." | 0.9 | 0.1 | 0.8 |
| "Our findings show the method works better." | 0.8 | 0.2 | 0.7 |
| "OMG I can't believe we won the match!!!" | 0.1 | 0.9 | 0.0 |

The first two sentences point in almost the same direction; the third points somewhere else.

### Cosine similarity

To measure "pointing in the same direction" we use **cosine similarity**:

$$\text{cosine}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

- the top is the **dot product**: multiply the numbers pairwise and add them up
- the bottom divides by the two vectors' lengths, so only the *direction* matters

The result is between -1 and 1. Close to 1 means very similar, around 0 means unrelated.

In [ ]:
import numpy as np

a = np.array([0.9, 0.1, 0.8])   # "The results indicate a significant improvement."
b = np.array([0.8, 0.2, 0.7])   # "Our findings show the method works better."
c = np.array([0.1, 0.9, 0.0])   # "OMG I can't believe we won the match!!!"

def cosine(x, y):
    return x @ y / (np.linalg.norm(x) * np.linalg.norm(y))

print(f"a vs b: {cosine(a, b):.3f}")
print(f"a vs c: {cosine(a, c):.3f}")

Real embedding models do the same thing with a few hundred features instead of three. Nobody chooses the features by hand: the model learns them from huge amounts of text. We can't name them, but the geometry works exactly the same way.

### A useful trick: vectors of length 1

If we scale every vector to length 1 first (**normalising** it), the bottom of the fraction becomes 1 × 1, and cosine similarity is just the dot product. Dot products are very fast to compute in bulk, which is why we will always normalise our embeddings.

In [ ]:
unit = lambda v: v / np.linalg.norm(v)
print(f"dot product of unit vectors: {unit(a) @ unit(b):.3f}  (same as the cosine above)")

## 2. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import torch
from transformers import AutoModel, AutoTokenizer

from ragdetect.data import LABEL_NAMES, load_mage
from ragdetect.embeddings import Embedder

## 3. Inside an embedding model, step by step

We'll use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a small, fast model trained specifically to produce good sentence embeddings. It turns any text into 384 numbers.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).eval()

**Step 1: split the text into tokens.** Models don't read whole words. They read *tokens*: common words stay whole, rarer words are split into pieces.

In [ ]:
print(tokenizer.tokenize("Retrieval makes detectors more robust."))

**Step 2: turn tokens into numbers and make a batch.** Each token is replaced by its ID in the model's vocabulary. Texts in a batch have different lengths, so shorter ones are filled up with a padding token. The **attention mask** records which positions are real (1) and which are padding (0).

In [ ]:
sentences = ["A short sentence.", "A noticeably longer sentence with quite a few more words in it."]
batch = tokenizer(sentences, padding=True, return_tensors="pt")
print("input_ids shape:", tuple(batch["input_ids"].shape))
print("attention mask:\n", batch["attention_mask"])

**Step 3: run the model.** It returns one vector *per token*.

In [ ]:
with torch.no_grad():
    token_vectors = model(**batch).last_hidden_state
print("token vectors:", tuple(token_vectors.shape), "= (texts, tokens, numbers per token)")

**Step 4: average the token vectors (mean pooling).** We want one vector per *text*, so we average its token vectors. Padding positions are filler, so we leave them out of the average using the attention mask.

In [ ]:
mask = batch["attention_mask"].unsqueeze(-1).float()
text_vectors = (token_vectors * mask).sum(dim=1) / mask.sum(dim=1)
print("text vectors:", tuple(text_vectors.shape))

**Step 5: normalise to length 1**, so dot products are cosine similarities.

In [ ]:
text_vectors = torch.nn.functional.normalize(text_vectors, dim=1).numpy()
print("lengths:", np.linalg.norm(text_vectors, axis=1))

That's all an embedding is. These five steps are wrapped up in `Embedder` in `ragdetect/embeddings.py`. Let's check it gives the same answer:

In [ ]:
embedder = Embedder(model_name)
same = np.allclose(embedder.encode(sentences), text_vectors, atol=1e-4)
print("Embedder matches our step-by-step version:", same)

## 4. Which sentences are similar?

Let's embed a few sentences on different topics and compare every pair.

In [ ]:
examples = [
    "Preheat the oven and whisk the eggs with sugar.",
    "Bake the cake for thirty minutes until golden.",
    "The striker scored twice in the second half.",
    "The home team won the league on the final day.",
    "We fine-tune a transformer model for text classification.",
]
vectors = embedder.encode(examples)
similarity = vectors @ vectors.T   # every pair at once: dot products of unit vectors

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(similarity, cmap="viridis", vmin=0, vmax=1)
labels = [s[:28] + "…" for s in examples]
ax.set_xticks(range(len(examples)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(examples)), labels)
for i in range(len(examples)):
    for j in range(len(examples)):
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)
fig.colorbar(im)
plt.title("Cosine similarity between sentences")
plt.show()

Sentences on the same topic (baking, football) should score clearly higher with each other than with the rest, even when they share hardly any words. The model has learned something about *meaning*.

## 5. Embedding real data

Now let's embed 1,000 MAGE texts.

In [ ]:
mage = load_mage("test", n=1000)
mage_vectors = embedder.encode(mage["text"], show_progress=True)
print(mage_vectors.shape)

We can't draw 384 dimensions, but **PCA** (principal component analysis) can squash them down to the two directions along which the points vary the most. It loses information, but it gives us a rough picture.

We'll draw the same points twice: coloured by label (human or machine), and coloured by the original dataset the text came from (the first part of `source`, e.g. `cmv` for Reddit's ChangeMyView).

In [ ]:
from sklearn.decomposition import PCA

points = PCA(n_components=2, random_state=0).fit_transform(mage_vectors)
origin = mage["source"].str.split("_").str[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, name in enumerate(LABEL_NAMES):
    keep = (mage["label"] == label).to_numpy()
    axes[0].scatter(points[keep, 0], points[keep, 1], s=6, alpha=0.6, label=name)
axes[0].set_title("Coloured by label")
axes[0].legend()

for name in origin.value_counts().index[:10]:
    keep = (origin == name).to_numpy()
    axes[1].scatter(points[keep, 0], points[keep, 1], s=6, alpha=0.6, label=name)
axes[1].set_title("Coloured by original dataset (10 largest)")
axes[1].legend(markerscale=3, fontsize=8)
plt.show()

**Look at the two pictures. Which one shows clearer groups?**

<details>
<summary>Discussion</summary>

Usually the right-hand picture (by dataset) shows much clearer groups. General-purpose embedding models like MiniLM are trained to put texts with similar **meaning or topic** close together. Whether a person or a machine wrote the text is a much subtler signal, and it isn't what this model was trained to capture.

This matters for us. In lesson 4 we'll use these embeddings to find "similar" texts, and "similar" will mostly mean "about the same thing". In lesson 6 we'll also try embeddings from a model that *was* trained to detect machine text, and compare.
</details>

## Check your understanding

<details>
<summary>1. Why do we normalise embeddings to length 1?</summary>

Because then the dot product of two embeddings equals their cosine similarity, and dot products for thousands of vectors can be computed in one fast matrix multiplication. We'll rely on this in lesson 3.
</details>

<details>
<summary>2. Why does mean pooling ignore padding tokens?</summary>

Padding is filler added only so texts in a batch have equal length. If we averaged it in, a text's embedding would change depending on which other texts happened to be in the same batch.
</details>

<details>
<summary>3. Two texts have cosine similarity 0.95. Another pair has 0.10. What does that tell you?</summary>

The first pair points in nearly the same direction in embedding space, so the model thinks they are very alike (usually in topic or meaning). The second pair is close to unrelated. It says nothing directly about who wrote them.
</details>

## Try it yourself

1. Write two sentences that mean the same thing but share almost no words, and one that shares many words but means something different. Which pair does the model think is more similar?
2. Swap in a bigger model, such as `sentence-transformers/all-mpnet-base-v2` (768 numbers per text). Does the PCA picture change?

## Summary

- An **embedding** is a vector that represents a text; similar texts get similar vectors.
- **Cosine similarity** compares directions. For unit-length vectors it is just the dot product.
- An embedding model goes: text → tokens → token vectors → mean pooling → normalise.
- General-purpose embeddings group texts mainly by **topic**, not by who wrote them.

**Next:** [Lesson 3 · Similarity search](03_similarity_search.ipynb), where we find a text's nearest neighbours among thousands.